In [1]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import col, upper, current_date
from awsglue.dynamicframe import DynamicFrame

# Initialize Glue context and Spark context
sc = SparkContext.getOrCreate() #get or create, important for jupyter.
glueContext = GlueContext(sc)
spark = glueContext.spark_session
# job = Job(glueContext)
# job.init(args['JOB_NAME'], args)
print("job started successfully")
# Point S3 to MinIO
sc._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://minio:9000")
sc._jsc.hadoopConfiguration().set("fs.s3a.access.key", "minioadmin")
sc._jsc.hadoopConfiguration().set("fs.s3a.secret.key", "minioadmin")
sc._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
sc._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

# Force Spark to use only the keys above, not AWS credential chain
sc._jsc.hadoopConfiguration().set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
sc._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "false")

print("minio set up completed")

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/home/glue_user/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/spark/jars/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/aws-glue-libs/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/aws-glue-libs/jars/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/glue_user/spark/python/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getO

job started successfully
minio set up completed


In [2]:
try:
    args = getResolvedOptions(sys.argv, ['JOB_NAME', 'input_path'])
except Exception:
    # Local fallback
    args = {
        'JOB_NAME': 'local_job',
        'input_path': 's3a://glue-spark-etl-example-source/raw/orders/orders6.csv',
        'output_path': 's3a://glue-spark-etl-example-target/silver/orders/',
    }

input_path = args['input_path']
output_path = args['output_path']

In [3]:
df = spark.read.csv("s3a://glue-spark-etl-example-source/raw/orders/orders6.csv", header=True)
df.show(5)

26/04/26 10:13:33 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+--------+--------------+----------+-------+----------+
|order_id| customer_name|   product| amount|order_date|
+--------+--------------+----------+-------+----------+
|ORD-0001| sarah johnson|    Webcam| 132.63|2024-11-17|
|ORD-0002|   emily davis|     Mouse|1227.21|2024-05-13|
|ORD-0003|    john smith|Headphones| 476.62|2024-03-22|
|ORD-0004|michael wilson|  Keyboard| 989.11|2024-10-11|
|ORD-0005|    john smith|  Keyboard|1916.09|2024-04-20|
+--------+--------------+----------+-------+----------+
only showing top 5 rows



In [4]:
datasource = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [input_path]},
    format="csv",
    format_options={"withHeader": True, "separator": ","}
)
datasource.show(5)

{"order_id": "ORD-0001", "customer_name": "sarah johnson", "product": "Webcam", "amount": "132.63", "order_date": "2024-11-17"}
{"order_id": "ORD-0002", "customer_name": "emily davis", "product": "Mouse", "amount": "1227.21", "order_date": "2024-05-13"}
{"order_id": "ORD-0003", "customer_name": "john smith", "product": "Headphones", "amount": "476.62", "order_date": "2024-03-22"}
{"order_id": "ORD-0004", "customer_name": "michael wilson", "product": "Keyboard", "amount": "989.11", "order_date": "2024-10-11"}
{"order_id": "ORD-0005", "customer_name": "john smith", "product": "Keyboard", "amount": "1916.09", "order_date": "2024-04-20"}


In [5]:
print(f"Records read: {datasource.count()}")
datasource.printSchema()

Records read: 104
root
|-- order_id: string
|-- customer_name: string
|-- product: string
|-- amount: string
|-- order_date: string



In [6]:
# ── Convert DynamicFrame to Spark DataFrame for transformations
df = datasource.toDF()

/home/glue_user/spark/python/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


In [7]:
df.show(5)

+--------+--------------+----------+-------+----------+
|order_id| customer_name|   product| amount|order_date|
+--------+--------------+----------+-------+----------+
|ORD-0001| sarah johnson|    Webcam| 132.63|2024-11-17|
|ORD-0002|   emily davis|     Mouse|1227.21|2024-05-13|
|ORD-0003|    john smith|Headphones| 476.62|2024-03-22|
|ORD-0004|michael wilson|  Keyboard| 989.11|2024-10-11|
|ORD-0005|    john smith|  Keyboard|1916.09|2024-04-20|
+--------+--------------+----------+-------+----------+
only showing top 5 rows



In [8]:
# ── Apply transformations ─────────────────────────────────────
df_transformed = df \
    .withColumn("customer_name", upper(col("customer_name"))) \
    .withColumn("processed_date", current_date()) \
    .filter(col("amount").cast("double") > 0) \
    .dropDuplicates(["order_id"])

df_transformed = df_transformed.repartition(2)

print(f"Records after transformation: {df_transformed.count()}")

Records after transformation: 100


In [9]:
# ── Convert back to DynamicFrame ──────────────────────────────
from awsglue.dynamicframe import DynamicFrame
output_frame = DynamicFrame.fromDF(df_transformed, glueContext, "output_frame")
output_frame.show(2)

{"order_id": "ORD-0018", "customer_name": "EMILY DAVIS", "product": "Webcam", "amount": "1469.56", "order_date": "2024-05-20", "processed_date": 2026-04-26}
{"order_id": "ORD-0079", "customer_name": "JOHN SMITH", "product": "Tablet", "amount": "1960.27", "order_date": "2024-11-19", "processed_date": 2026-04-26}


In [10]:
# ── Write output as Parquet to S3 ─────────────────────────────
glueContext.write_dynamic_frame.from_options(
    frame=output_frame,
    connection_type="s3",
    connection_options={
        "path": output_path,
        "partitionKeys": ["processed_date"]
    },
    format="parquet"
)

print("Job completed successfully")

Job completed successfully


In [11]:
# spark.stop()
# print("spark application stopped")